### Pré-processamento Textual (spaCy)

In [96]:
#pip install spacy
#!python -m spacy download pt_core_news_sm

In [97]:
import spacy
from collections import Counter, defaultdict
from datetime import datetime
from statistics import median
from typing import Any
from zoneinfo import ZoneInfo

nlp = spacy.load("pt_core_news_sm")
APP_TIMEZONE = ZoneInfo("America/Sao_Paulo")
MATTR_WINDOW_SIZE = 20

PERIOD_WINDOWS = {
    "baseline": ("2026-05-01", "2026-05-30"),
    "reanalise": ("2026-05-31", "2026-06-14"),
}
BASELINE_PERIOD_NAME = "baseline"
BASELINE_START_DATE, BASELINE_END_DATE = PERIOD_WINDOWS[BASELINE_PERIOD_NAME]


def normalize_text(text: str) -> str:
    return " ".join((text or "").split())


def parse_message_datetime(value: str) -> datetime:
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=APP_TIMEZONE)
    return parsed.astimezone(APP_TIMEZONE)


def get_date(created_at: str) -> str:
    return parse_message_datetime(created_at).date().isoformat()


def is_valid_word(token) -> bool:
    return (
        not token.is_space
        and not token.is_punct
        and not token.like_url
        and not token.like_email
    )


def get_words(doc):
    return [token for token in doc if is_valid_word(token)]


def get_lemmas(words):
    return [
        token.lemma_.lower()
        for token in words
        if not token.is_stop and token.lemma_.strip()
    ]


def safe_divide(numerator: float, denominator: float) -> float:
    if denominator == 0:
        return 0.0
    return round(numerator / denominator, 4)


def calculate_mattr(lemmas: list[str], window_size: int = MATTR_WINDOW_SIZE) -> float | None:
    """Calcula Moving-Average Type-Token Ratio (MATTR) em janelas lexicais fixas."""
    if window_size <= 0:
        raise ValueError("window_size deve ser maior que zero.")
    if len(lemmas) < window_size:
        return None

    window_ttrs = [
        len(set(lemmas[start:start + window_size])) / window_size
        for start in range(len(lemmas) - window_size + 1)
    ]
    return round(sum(window_ttrs) / len(window_ttrs), 4)


def _period_name_for_date(day: str, period_windows: dict[str, tuple[str, str]]) -> str | None:
    for period_name, (start_date, end_date) in period_windows.items():
        if start_date <= day <= end_date:
            return period_name
    return None


def extract_text_metrics(message_id: str, user_id: str, text: str) -> dict[str, Any]:
    text = normalize_text(text)
    doc = nlp(text)
    words = get_words(doc)
    lemmas = get_lemmas(words)

    pos_counter = Counter(token.pos_ for token in words)
    lemma_counter = Counter(lemmas)
    word_count = len(words)
    lemma_count = len(lemmas)

    return {
        "messageId": message_id,
        "userId": user_id,
        "text": text,
        "metrics": {
            "basicLinguistic": {
                "charCount": len(text),
                "tokenCount": len(doc),
                "wordCount": word_count,
                "sentenceCount": len(list(doc.sents)),
                "avgTokenLength": safe_divide(
                    sum(len(token.text) for token in words),
                    word_count,
                ),
                "lemmaCount": lemma_count,
                "uniqueLemmaCount": len(set(lemmas)),
                "stopwordCount": sum(1 for token in words if token.is_stop),
                "nounCount": pos_counter.get("NOUN", 0),
                "properNounCount": pos_counter.get("PROPN", 0),
                "verbCount": pos_counter.get("VERB", 0),
                "auxVerbCount": pos_counter.get("AUX", 0),
                "adjectiveCount": pos_counter.get("ADJ", 0),
                "adverbCount": pos_counter.get("ADV", 0),
                "pronounCount": pos_counter.get("PRON", 0),
                "topLemmas": dict(lemma_counter.most_common(10)),
            }
        },
    }


def extract_many_text_metrics(messages: list[dict]) -> list[dict]:
    return [
        extract_text_metrics(
            message_id=message["messageId"],
            user_id=message["userId"],
            text=message["text"],
        ) | {"createdAt": message["createdAt"]}
        for message in messages
    ]


def aggregate_by_day(results: list[dict]) -> list[dict]:
    grouped = defaultdict(list)

    for item in results:
        grouped[(item["userId"], get_date(item["createdAt"]))].append(item)

    daily_results = []

    for (user_id, day), items in grouped.items():
        metrics = [item["metrics"] for item in items]
        message_count = len(items)
        total_words = sum(m["basicLinguistic"]["wordCount"] for m in metrics)
        total_chars = sum(m["basicLinguistic"]["charCount"] for m in metrics)
        total_sentences = sum(m["basicLinguistic"]["sentenceCount"] for m in metrics)
        total_lemmas = sum(m["basicLinguistic"]["lemmaCount"] for m in metrics)

        # Diversidade lexical não é calculada diariamente. O MATTR é calculado no
        # corpus agregado de cada ciclo para reduzir a dependência do tamanho do texto.
        daily_results.append({
            "userId": user_id,
            "date": day,
            "basicLinguistic": {
                "messageCount": message_count,
                "totalWords": total_words,
                "totalChars": total_chars,
                "totalSentences": total_sentences,
                "totalLemmas": total_lemmas,
                "avgWordsPerMessage": round(total_words / message_count, 2) if message_count else None,
                "avgCharsPerMessage": round(total_chars / message_count, 2) if message_count else None,
                "avgSentencesPerMessage": round(total_sentences / message_count, 2) if message_count else None,
                "totalNouns": sum(m["basicLinguistic"]["nounCount"] for m in metrics),
                "totalVerbs": sum(m["basicLinguistic"]["verbCount"] for m in metrics),
                "totalAdjectives": sum(m["basicLinguistic"]["adjectiveCount"] for m in metrics),
                "totalAdverbs": sum(m["basicLinguistic"]["adverbCount"] for m in metrics),
            },
        })

    return sorted(daily_results, key=lambda item: (item["userId"], item["date"]))


def aggregate_lexical_diversity_by_period(
    results: list[dict],
    period_windows: dict[str, tuple[str, str]],
    window_size: int = MATTR_WINDOW_SIZE,
    baseline_period_name: str = BASELINE_PERIOD_NAME,
) -> list[dict]:
    """Calcula MATTR sobre todo o corpus textual de cada usuário em cada ciclo."""
    grouped: dict[tuple[str, str], list[dict]] = defaultdict(list)

    for item in results:
        day = get_date(item["createdAt"])
        period_name = _period_name_for_date(day, period_windows)
        if period_name is not None:
            grouped[(item["userId"], period_name)].append(item)

    period_results: list[dict] = []

    for (user_id, period_name), items in grouped.items():
        items = sorted(items, key=lambda item: parse_message_datetime(item["createdAt"]))
        period_text = " ".join(item["text"] for item in items)
        period_doc = nlp(period_text)
        lemmas = get_lemmas(get_words(period_doc))
        total_lemmas = len(lemmas)
        mattr = calculate_mattr(lemmas, window_size=window_size)
        start_date, end_date = period_windows[period_name]

        period_results.append({
            "userId": user_id,
            "period": period_name,
            "startDate": start_date,
            "endDate": end_date,
            "lexicalDiversityMattr": mattr,
            "mattrWindowSize": window_size,
            "totalLemmas": total_lemmas,
            "mattrWindowCount": max(total_lemmas - window_size + 1, 0),
            "sampleSufficient": mattr is not None,
        })

    baseline_mattr_by_user = {
        item["userId"]: item.get("lexicalDiversityMattr")
        for item in period_results
        if item.get("period") == baseline_period_name
    }

    for item in period_results:
        item["baselineLexicalDiversityMattr"] = baseline_mattr_by_user.get(item["userId"])

    return sorted(period_results, key=lambda item: (item["userId"], item["startDate"]))


def _median_and_mad(values: list[float]) -> dict[str, float | int | None]:
    """Referência robusta do baseline: mediana e MAD (median absolute deviation)."""
    valid = [float(value) for value in values if isinstance(value, (int, float))]
    if not valid:
        return {"median": None, "mad": None, "sampleDays": 0}

    center = median(valid)
    mad = median([abs(value - center) for value in valid])

    return {
        "median": round(center, 4),
        "mad": round(mad, 4),
        "sampleDays": len(valid),
    }


def add_linguistic_baseline_reference(
    daily_results: list[dict],
    baseline_start_date: str,
    baseline_end_date: str,
) -> list[dict]:
    """
    Anexa a cada dia a referência linguística do próprio usuário no baseline.

    As medidas linguísticas deixam de ser avaliadas por faixas universais de
    "bom/ruim" e passam a indicar estabilidade ou desvio do padrão individual.
    """
    grouped = defaultdict(list)
    for item in daily_results:
        grouped[item["userId"]].append(item)

    references: dict[str, dict[str, dict]] = {}

    for user_id, items in grouped.items():
        baseline_items = [
            item["basicLinguistic"]
            for item in items
            if baseline_start_date <= item["date"] <= baseline_end_date
        ]

        references[user_id] = {
            "avgWordsPerMessage": _median_and_mad([
                item.get("avgWordsPerMessage")
                for item in baseline_items
            ]),
            "avgSentencesPerMessage": _median_and_mad([
                item.get("avgSentencesPerMessage")
                for item in baseline_items
            ]),
        }

    enriched = []
    for item in daily_results:
        copy_item = {
            **item,
            "basicLinguistic": {
                **item["basicLinguistic"],
                "baselineReference": references.get(item["userId"], {}),
            },
        }
        enriched.append(copy_item)

    return sorted(enriched, key=lambda item: (item["userId"], item["date"]))


if __name__ == "__main__":
    import json

    with open("../Logs/mensagens_simuladas_4perfis.json", "r", encoding="utf-8") as file:
        messages = json.load(file)

    results = extract_many_text_metrics(messages)
    basic_linguistic_results = aggregate_by_day(results)
    basic_linguistic_results = add_linguistic_baseline_reference(
        basic_linguistic_results,
        baseline_start_date=BASELINE_START_DATE,
        baseline_end_date=BASELINE_END_DATE,
    )
    lexical_diversity_period_results = aggregate_lexical_diversity_by_period(
        results,
        period_windows=PERIOD_WINDOWS,
        window_size=MATTR_WINDOW_SIZE,
    )

    print("########## INDIVIDUAIS ##########")
    for item in results:
        print(item)

    print()
    print("########## AGRUPADOS ##########")
    for item in basic_linguistic_results:
        print(item)

########## INDIVIDUAIS ##########
{'messageId': 'msg-0001', 'userId': 'user-001', 'text': 'Encontrei um comportamento estranho em tratamento de logs: um erro está sendo tratado como sucesso. Consegui reproduzir e agora estou isolando a causa. Se confirmar o que estou vendo, a correção deve ser pequena.', 'metrics': {'basicLinguistic': {'charCount': 212, 'tokenCount': 39, 'wordCount': 34, 'sentenceCount': 3, 'avgTokenLength': 5.1176, 'lemmaCount': 17, 'uniqueLemmaCount': 17, 'stopwordCount': 17, 'nounCount': 7, 'properNounCount': 0, 'verbCount': 8, 'auxVerbCount': 5, 'adjectiveCount': 2, 'adverbCount': 1, 'pronounCount': 2, 'topLemmas': {'encontrei': 1, 'comportamento': 1, 'estranho': 1, 'tratamento': 1, 'logs': 1, 'erro': 1, 'ser': 1, 'tratar': 1, 'sucesso': 1, 'consegui': 1}}}, 'createdAt': '2026-05-01T09:47:00'}
{'messageId': 'msg-0002', 'userId': 'user-001', 'text': 'O trabalho em API de pedidos ficou bloqueado porque uma dependência de outro card ainda não foi liberada. Já sinalize

### Análise de Sentimento e Polaridade (pysentimiento e Transformers)

In [98]:
#pip install pysentimiento transformers torch

In [99]:
from pysentimiento import create_analyzer
from collections import Counter, defaultdict
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

sentiment_analyzer = create_analyzer(task="sentiment", lang="pt")

def extract_sentiment_metrics(message_id: str, user_id: str, text: str) -> dict:
    result = sentiment_analyzer.predict(text)

    return {
        "messageId": message_id,
        "userId": user_id,
        "text": text,
        "metrics": {
            "sentimentAnalysis": {
                "sentimentLabel": result.output,
                "positiveScore": round(result.probas.get("POS", 0.0), 4),
                "negativeScore": round(result.probas.get("NEG", 0.0), 4),
                "neutralScore": round(result.probas.get("NEU", 0.0), 4),
                "sentimentProbabilities": {
                    label: round(score, 4)
                    for label, score in result.probas.items()
                }
            }
        }
    }

def extract_many_sentiment_metrics(messages: list[dict]) -> list[dict]:
    return [
        extract_sentiment_metrics(
            message_id=message["messageId"],
            user_id=message["userId"],
            text=message["text"]
        ) | {
            "createdAt": message["createdAt"]
        }
        for message in messages
    ]

def get_dominant_sentiment(label_counter: Counter) -> str:
    if not label_counter:
        return "UNKNOWN"

    most_common = label_counter.most_common()
    highest_count = most_common[0][1]

    top_labels = [
        label
        for label, count in most_common
        if count == highest_count
    ]

    if len(top_labels) > 1:
        return "MIXED"

    return top_labels[0]

def aggregate_sentiment_by_day(results: list[dict]) -> list[dict]:
    grouped = defaultdict(list)

    for item in results:
        day = get_date(item["createdAt"])
        grouped[(item["userId"], day)].append(item)

    daily_results = []

    for (user_id, day), items in grouped.items():
        metrics = [
            item["metrics"]["sentimentAnalysis"]
            for item in items
        ]

        message_count = len(items)

        labels = [
            m["sentimentLabel"]
            for m in metrics
        ]

        label_counter = Counter(labels)

        avg_positive = sum(m["positiveScore"] for m in metrics) / message_count
        avg_negative = sum(m["negativeScore"] for m in metrics) / message_count
        avg_neutral = sum(m["neutralScore"] for m in metrics) / message_count

        daily_results.append({
            "userId": user_id,
            "date": day,
            "sentimentAnalysis": {
                "messageCount": message_count,
                "avgPositiveScore": round(avg_positive, 4),
                "avgNegativeScore": round(avg_negative, 4),
                "avgNeutralScore": round(avg_neutral, 4),
                "sentimentLabelCounts": dict(label_counter),
                "dominantSentiment": get_dominant_sentiment(label_counter)
            }
        })

    return daily_results

if __name__ == "__main__":
    sentiment_results = extract_many_sentiment_metrics(messages)
    sentiment_analysis_results = aggregate_sentiment_by_day(sentiment_results)

    print("########## SENTIMENTO INDIVIDUAL ##########")
    for item in sentiment_results:
        print(item)

    print()
    print("########## SENTIMENTO AGRUPADO ##########")
        
    for item in sentiment_analysis_results:
        print(item)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6190.79it/s]


########## SENTIMENTO INDIVIDUAL ##########
{'messageId': 'msg-0001', 'userId': 'user-001', 'text': 'Encontrei um comportamento estranho em tratamento de logs: um erro está sendo tratado como sucesso. Consegui reproduzir e agora estou isolando a causa. Se confirmar o que estou vendo, a correção deve ser pequena.', 'metrics': {'sentimentAnalysis': {'sentimentLabel': 'NEG', 'positiveScore': 0.0206, 'negativeScore': 0.6716, 'neutralScore': 0.3077, 'sentimentProbabilities': {'NEG': 0.6716, 'NEU': 0.3077, 'POS': 0.0206}}}, 'createdAt': '2026-05-01T09:47:00'}
{'messageId': 'msg-0002', 'userId': 'user-001', 'text': 'O trabalho em API de pedidos ficou bloqueado porque uma dependência de outro card ainda não foi liberada. Já sinalizei e vou pegar uma tarefa menor enquanto espero.', 'metrics': {'sentimentAnalysis': {'sentimentLabel': 'NEU', 'positiveScore': 0.0202, 'negativeScore': 0.1007, 'neutralScore': 0.8791, 'sentimentProbabilities': {'NEG': 0.1007, 'NEU': 0.8791, 'POS': 0.0202}}}, 'created

### Representação Semântica (Sentence Transformers)

In [100]:
#pip install sentence-transformers

In [101]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
from typing import Any
import numpy as np

semantic_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


def extract_semantic_metrics(message_id: str, user_id: str, text: str) -> dict[str, Any]:
    embedding = semantic_model.encode(text)

    return {
        "messageId": message_id,
        "userId": user_id,
        "text": text,
        "metrics": {
            "semanticRepresentation": {
                "embedding": embedding.tolist(),
                "embeddingSize": len(embedding),
            }
        },
    }


def extract_many_semantic_metrics(messages: list[dict]) -> list[dict]:
    return [
        extract_semantic_metrics(
            message_id=message["messageId"],
            user_id=message["userId"],
            text=message["text"],
        ) | {"createdAt": message["createdAt"]}
        for message in messages
    ]


def group_texts_by_day(messages: list[dict]) -> list[dict]:
    grouped = defaultdict(list)

    for message in messages:
        grouped[(message["userId"], get_date(message["createdAt"]))].append(message["text"])

    return [
        {
            "userId": user_id,
            "date": day,
            "text": " ".join(texts),
        }
        for (user_id, day), texts in grouped.items()
    ]


def extract_daily_semantic_embeddings(messages: list[dict]) -> list[dict]:
    results = []

    for item in group_texts_by_day(messages):
        embedding = semantic_model.encode(item["text"])
        results.append({
            "userId": item["userId"],
            "date": item["date"],
            "text": item["text"],
            "semanticRepresentation": {
                "embedding": embedding.tolist(),
                "embeddingSize": len(embedding),
            },
        })

    return results


def classify_semantic_shift(distance: float | None) -> str:
    if distance is None:
        return "NO_PREVIOUS_OBSERVATION"
    if distance >= 0.50:
        return "HIGH"
    if distance >= 0.35:
        return "MODERATE"
    return "LOW"


def _cosine_similarity(first: np.ndarray, second: np.ndarray) -> float:
    similarity = cosine_similarity(
        first.reshape(1, -1),
        second.reshape(1, -1),
    )[0][0]
    return round(float(similarity), 4)


def calculate_semantic_shift(
    daily_results: list[dict],
    baseline_start_date: str | None = None,
    baseline_end_date: str | None = None,
) -> list[dict]:
    """
    Produz duas referências distintas:
    1) similaridade com a observação anterior (estabilidade local);
    2) similaridade com o centróide semântico do baseline (desvio de patamar).
    """
    grouped = defaultdict(list)
    for item in daily_results:
        grouped[item["userId"]].append(item)

    shift_results = []

    for user_id, items in grouped.items():
        items = sorted(items, key=lambda item: item["date"])

        baseline_items = [
            item
            for item in items
            if (baseline_start_date is None or item["date"] >= baseline_start_date)
            and (baseline_end_date is None or item["date"] <= baseline_end_date)
        ]

        baseline_centroid = None
        if baseline_items:
            baseline_embeddings = np.array([
                item["semanticRepresentation"]["embedding"]
                for item in baseline_items
            ])
            baseline_centroid = baseline_embeddings.mean(axis=0)

        for index, current_item in enumerate(items):
            current_embedding = np.array(
                current_item["semanticRepresentation"]["embedding"]
            )

            previous_date = None
            similarity_previous = None
            distance_previous = None

            if index > 0:
                previous_item = items[index - 1]
                previous_embedding = np.array(
                    previous_item["semanticRepresentation"]["embedding"]
                )
                previous_date = previous_item["date"]
                similarity_previous = _cosine_similarity(current_embedding, previous_embedding)
                distance_previous = round(1.0 - similarity_previous, 4)

            similarity_baseline = None
            distance_baseline = None
            if baseline_centroid is not None:
                similarity_baseline = _cosine_similarity(current_embedding, baseline_centroid)
                distance_baseline = round(1.0 - similarity_baseline, 4)

            shift_results.append({
                "userId": user_id,
                "date": current_item["date"],
                "semanticShift": {
                    "previousDate": previous_date,
                    "similarityToPreviousDay": similarity_previous,
                    "distanceToPreviousDay": distance_previous,
                    "semanticPreviousDayChangeLevel": classify_semantic_shift(distance_previous),
                    "similarityToBaseline": similarity_baseline,
                    "distanceToBaseline": distance_baseline,
                    "semanticBaselineChangeLevel": classify_semantic_shift(distance_baseline),
                },
            })

    return sorted(shift_results, key=lambda item: (item["userId"], item["date"]))


if __name__ == "__main__":
    daily_semantic_results = extract_daily_semantic_embeddings(messages)
    semantic_shift_results = calculate_semantic_shift(
        daily_semantic_results,
        baseline_start_date=BASELINE_START_DATE,
        baseline_end_date=BASELINE_END_DATE,
    )

    print("########## REPRESENTAÇÃO SEMÂNTICA DIÁRIA ##########")
    for item in daily_semantic_results:
        print({
            "userId": item["userId"],
            "date": item["date"],
            "embeddingSize": item["semanticRepresentation"]["embeddingSize"],
            "text": item["text"],
        })

    print()
    print("########## MUDANÇA SEMÂNTICA ENTRE DIAS E BASELINE ##########")
    for item in semantic_shift_results:
        print(item)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4716.08it/s]


########## REPRESENTAÇÃO SEMÂNTICA DIÁRIA ##########
{'userId': 'user-001', 'date': '2026-05-01', 'embeddingSize': 384, 'text': 'Encontrei um comportamento estranho em tratamento de logs: um erro está sendo tratado como sucesso. Consegui reproduzir e agora estou isolando a causa. Se confirmar o que estou vendo, a correção deve ser pequena. O trabalho em API de pedidos ficou bloqueado porque uma dependência de outro card ainda não foi liberada. Já sinalizei e vou pegar uma tarefa menor enquanto espero. Vou começar o dia em webhook de atualização. Terminei a parte prevista de endpoint de relatórios. Antes de encerrar, conferi o fluxo principal e atualizei o card. Minha prioridade agora é endpoint de relatórios. Pareceu melhor pausar endpoint de relatórios por agora, já que a resposta de um serviço externo veio diferente do esperado. Assim que tiver retorno eu retomo.'}
{'userId': 'user-001', 'date': '2026-05-02', 'embeddingSize': 384, 'text': 'Teve retorno em integração de notificações: 

In [102]:
import pandas as pd

PROFILE_NAMES = {
    "user-001": "Perfil A",
    "user-002": "Perfil B",
    "user-003": "Perfil C",
    "user-004": "Perfil D",
}

PERIOD_NAMES = {
    "baseline": "Baseline (30 dias)",
    "reanalise": "Reanálise (15 dias)",
}

mattr_table = pd.DataFrame([
    {
        "Perfil": PROFILE_NAMES.get(item["userId"], item["userId"]),
        "Período": PERIOD_NAMES.get(item["period"], item["period"]),
        "MATTR": item.get("lexicalDiversityMattr"),
        "MATTR Baseline": item.get("baselineLexicalDiversityMattr"),
        "Δ MATTR vs. Baseline": (
            round(
                item["lexicalDiversityMattr"]
                - item["baselineLexicalDiversityMattr"],
                4,
            )
            if item.get("lexicalDiversityMattr") is not None
            and item.get("baselineLexicalDiversityMattr") is not None
            else None
        ),
        "Total de Lemas": item.get("totalLemmas"),
        "Tamanho da Janela": item.get("mattrWindowSize"),
        "Janelas MATTR": item.get("mattrWindowCount"),
        "Amostra Suficiente": item.get("sampleSufficient"),
    }
    for item in lexical_diversity_period_results
])

profile_order = ["Perfil A", "Perfil B", "Perfil C", "Perfil D"]
period_order = ["Baseline (30 dias)", "Reanálise (15 dias)"]

mattr_table["Perfil"] = pd.Categorical(
    mattr_table["Perfil"],
    categories=profile_order,
    ordered=True,
)

mattr_table["Período"] = pd.Categorical(
    mattr_table["Período"],
    categories=period_order,
    ordered=True,
)

mattr_table = (
    mattr_table
    .sort_values(["Perfil", "Período"])
    .reset_index(drop=True)
)

display(mattr_table)

,Perfil,Período,MATTR,MATTR Baseline,Δ MATTR vs. Baseline,Total de Lemas,Tamanho da Janela,Janelas MATTR,Amostra Suficiente
0,Perfil A,Baseline (30 dias),0.9013,0.9013,0.0000,1285,20,1266,True
1,Perfil A,Reanálise (15 dias),0.9265,0.9013,0.0252,718,20,699,True
2,Perfil B,Baseline (30 dias),0.9188,0.9188,0.0000,1642,20,1623,True
3,Perfil B,Reanálise (15 dias),0.8605,0.9188,-0.0583,516,20,497,True
4,Perfil C,Baseline (30 dias),0.9309,0.9309,0.0000,1781,20,1762,True
5,Perfil C,Reanálise (15 dias),0.9292,0.9309,-0.0017,883,20,864,True
6,Perfil D,Baseline (30 dias),0.9104,0.9104,0.0000,1547,20,1528,True
7,Perfil D,Reanálise (15 dias),0.8645,0.9104,-0.0459,494,20,475,True


In [103]:
from collections import defaultdict
import json

def group_by_user(results: list[dict]) -> dict:
    grouped = defaultdict(list)

    for item in results:
        user_id = item["userId"]

        item_without_user = {
            key: value
            for key, value in item.items()
            if key != "userId"
        }

        grouped[user_id].append(item_without_user)

    return dict(grouped)

indicadores_chat = {
    "basicLinguistic": group_by_user(basic_linguistic_results),
    "lexicalDiversityByPeriod": group_by_user(lexical_diversity_period_results),
    "sentimentAnalysis": group_by_user(sentiment_analysis_results),
    "semanticShift": group_by_user(semantic_shift_results),
}

with open("../Logs/indicadores_chat.json", "w", encoding="utf-8") as file:
    json.dump(indicadores_chat, file, ensure_ascii=False)